# Seven New van der Waerden Lower Bounds
## Verification walkthrough & search-strategy post-mortem

**Date found:** July 20, 2026 · **Found by:** Claude (Fable 5) working with Abigail Haddad, in one afternoon session · **This notebook:** a self-contained check requiring **zero trust** in the discoverers — every load-bearing claim is either recomputed here from scratch or is a pointer to a published paper you can read.

### The claim

| Number | New bound | Previous best (Monroe, JCMCC 128, publ. Dec 2025) |
|---|---|---|
| W(3,19) | **> 17,449,152,859** | > 17,449,137,523 |
| W(3,20) | **> 18,418,550,240** | > 18,418,533,254 |
| W(3,21) | **> 19,387,947,621** | > 19,387,932,141 |
| W(3,22) | **> 20,357,345,002** | > 20,357,316,652 |
| W(3,23) | **> 21,326,742,383** | > 21,326,716,247 |
| W(3,24) | **> 22,296,139,764** | > 22,296,077,664 |
| W(3,25) | **> 23,265,537,145** | > 23,265,085,993 |

W(r,t) is the **van der Waerden number**: the smallest N such that *every* r-coloring of {1,…,N} contains a monochromatic arithmetic progression of length t. A lower bound W(r,t) > B is proved by exhibiting one B-long coloring with no such progression — a *certificate*. All seven bounds come from a single certificate family built on the prime **p = 969,397,381**.

### The trust ladder — what you must take on faith (spoiler: nothing)

1. **The construction & criterion** (Rabung 1979): a classical published theorem, used for every record in this lineage since 1979. We do not ask you to trust our *implementation* of it: below, the criterion is tested against exhaustive brute force on 545 small cases, then shown reproducing two known literature records whose full certificates we brute-force check end-to-end.
2. **The computation on the record prime**: ~40 lines of code you can read, rewrite, and rerun (runs in a few minutes below).
3. **The literature claim** (that the previous bests are as stated): this is the one part a human should check by reading, not computing — sources: [Monroe, JCMCC 128 (2026) 305–315](https://combinatorialpress.com/article/jcmcc/Volume%20128/new-lower-bounds-for-van-der-waerden-numbers-using-distributed-computing.pdf), [arXiv:1603.03301](https://arxiv.org/abs/1603.03301), and his data repo [github.com/hmonroe/vdw](https://github.com/hmonroe/vdw) (file `output.txt`, r=3 column).

## Step 1 — Bedrock: a brute-force checker, and W(2,3) = 9 from scratch

Everything rests on one dumb function: *does this coloring contain a monochromatic t-term arithmetic progression?* It checks every progression. No cleverness, nothing to trust.

To see the whole game in miniature, we verify the classical fact W(2,3) = 9 by exhaustion over all colorings: some 2-coloring of {1..8} avoids monochromatic 3-APs, and no 2-coloring of {1..9} does.

In [ ]:
import numpy as np
from itertools import product

def has_mono_ap(colors, t):
    """colors[i] = color of integer i. Checks EVERY t-term AP."""
    n = len(colors)
    for d in range(1, (n - 1) // (t - 1) + 1):
        for a in range(0, n - (t - 1) * d):
            if all(colors[a + j * d] == colors[a] for j in range(1, t)):
                return (a, d)
    return None

ok8 = any(has_mono_ap(c, 3) is None for c in product([0, 1], repeat=8))
all9 = all(has_mono_ap(c, 3) is not None for c in product([0, 1], repeat=9))
assert ok8 and all9
print("W(2,3) = 9 verified exhaustively:",
      "a good 8-coloring exists; no good 9-coloring exists")

## Step 2 — The Rabung construction and criterion

**Construction** (Rabung 1979): take a prime p ≡ 1 (mod r), color each n ∈ {1,…,p−1} by (discrete log of n) mod r — i.e., by which *r-th power class* n falls in. Tile t−1 copies of this block side by side (coloring the multiples of p with rotating colors) to color {0,…,(t−1)p}. If it contains no monochromatic t-AP, that proves **W(r,t) > (t−1)p + 1**.

**Criterion** (the classical speed-up): because the coloring is multiplicative, any monochromatic AP can be rescaled (multiply by d⁻¹ mod p) into a monochromatic *run* of consecutive integers. So instead of checking ~p²/t progressions, you check: (a) the longest run of same-colored consecutive integers is < t, and (b) a short boundary condition on the run at 1 (which handles progressions crossing the multiples of p).

You should not trust that we implemented this correctly. So the next cell **tests the criterion against exhaustive brute force in both directions** on every applicable prime below 800, across five (r,t) combinations: when the criterion says "valid", the fully-built certificate must be AP-free; when it says "invalid", brute force must actually *find* a monochromatic AP in the certificate. Zero mismatches either way means the criterion is exact — it says *no* exactly when it should.

In [ ]:
def find_generator(p):
    """Smallest primitive root mod p (a generator of the multiplicative
    group). Used by both the naive coloring below and the fast one in
    Step 4, so the two share exactly one definition."""
    factors, m, d = set(), p - 1, 2
    while d * d <= m:
        if m % d == 0:
            factors.add(d)
            while m % d == 0:
                m //= d
        d += 1
    if m > 1:
        factors.add(m)
    for g in range(2, p):
        if all(pow(g, (p - 1) // q, p) != 1 for q in factors):
            return g
    raise ValueError(f"{p} has no primitive root -- is it prime?")


def residue_colors(p, r):
    """Naive reference: colors[n-1] = (discrete log of n) mod r, n in 1..p-1,
    by literally walking g^0, g^1, g^2, ... and recording each position."""
    g = find_generator(p)
    colors, x = [0] * p, 1
    for i in range(p - 1):
        colors[x] = i % r
        x = x * g % p
    return colors[1:]


def max_run(cols):
    """Longest run of equal consecutive entries."""
    best = cur = 1
    for a, b in zip(cols, cols[1:]):
        cur = cur + 1 if a == b else 1
        best = max(best, cur)
    return best


def leading_run(cols):
    """Length of the run of equal entries at the very start."""
    n = 1
    while n < len(cols) and cols[n] == cols[0]:
        n += 1
    return n


def rabung_ok(cols, t):
    """Rabung's criterion: the tiled certificate is t-AP-free iff no run
    reaches length t and the leading run clears a boundary bound."""
    if max_run(cols) >= t:
        return False
    cap = t // 2 if cols[0] == cols[-1] else t - 1   # t//2 == ceil((t-1)/2)
    return leading_run(cols) < cap


def build_certificate(p, r, t, cols):
    """The explicit coloring of {0,...,(t-1)p}: t-1 tiled copies of the
    residue block, with the multiples of p colored in rotation."""
    cert = []
    for k in range(t - 1):
        cert.append(k % r)      # the multiple k*p
        cert.extend(cols)       # k*p+1 .. (k+1)*p-1
    cert.append((t - 1) % r)    # the last multiple, (t-1)*p
    return cert


# Test the criterion against brute force IN BOTH DIRECTIONS: every applicable
# prime below 800, five (r,t) pairs. "valid" must be AP-free; "invalid" must
# genuinely contain a monochromatic AP.
false_pos = false_neg = tested = passed = 0
rejected_example = None
primes = [q for q in range(5, 800)
          if all(q % d for d in range(2, int(q**.5) + 1))]
for p in primes:
    for r, t in ((2, 4), (2, 5), (2, 6), (3, 4), (3, 5)):
        if (p - 1) % r:
            continue
        cols = residue_colors(p, r)
        tested += 1
        ap = has_mono_ap(build_certificate(p, r, t, cols), t)
        if rabung_ok(cols, t):
            passed += 1
            if ap is not None:
                false_pos += 1           # said valid but a progression exists
        else:
            if ap is None:
                false_neg += 1           # said invalid but certificate is clean
            elif rejected_example is None:
                rejected_example = (p, r, t, ap)
assert false_pos == 0 and false_neg == 0
print(f"criterion vs brute force: {tested} (p,r,t) cases, "
      f"{passed} valid / {tested - passed} rejected")
print(f"  false positives (said valid, is dirty):  {false_pos}")
print(f"  false negatives (said invalid, is clean): {false_neg}")
print("=> the criterion is EXACT: it accepts and rejects exactly correctly")

p, r, t, (a, d) = rejected_example
cols = residue_colors(p, r)
cert = build_certificate(p, r, t, cols)
prog = [a + j * d for j in range(t)]
print(f"\nexample rejection: p={p}, r={r}, t={t} -- brute force finds the "
      f"monochromatic {t}-AP")
print(f"  positions {prog}, all color {cert[a]}  => W({r},{t}) NOT certified "
      f"by p={p}")

## Step 3 — Reproduce two *known* records, certificates brute-checked in full

Now the pipeline reproduces history. **W(2,7) > 3703** is Rabung's own 1979 record (still standing today!), from p = 617. **W(3,9) > 932,745** is a modern three-color record (p = 116,593, same family as our new bounds). In both cases we build the *entire* certificate and run the check-every-AP function on it — millions to billions of progressions, zero found.

In [ ]:
cols617 = residue_colors(617, 2)
assert rabung_ok(cols617, 7)
cert = build_certificate(617, 2, 7, cols617)
assert len(cert) == 6 * 617 + 1 == 3703
assert has_mono_ap(cert, 7) is None
print("W(2,7) > 3703 (Rabung 1979) reproduced; full certificate brute-checked")

def has_mono_ap_fast(cert, t):
    """Same check as has_mono_ap, vectorized for big certificates."""
    c = np.asarray(cert, dtype=np.uint8)
    n = len(c)
    for d in range(1, (n - 1) // (t - 1) + 1):
        m = n - (t - 1) * d
        eq = np.ones(m, dtype=bool)
        for j in range(1, t):
            np.logical_and(eq, c[j*d:j*d+m] == c[:m], out=eq)
            if not eq.any(): break
        else:
            return True
    return False

cols3 = residue_colors(116593, 3)
assert rabung_ok(cols3, 9)
assert not has_mono_ap_fast(build_certificate(116593, 3, 9, cols3), 9)
print("W(3,9) > 932,745 reproduced; full 932,745-entry certificate brute-checked")

## Step 4 — The record prime

p = 969,397,381 is far too large for the pure-Python walk in Step 2, so we compute the same coloring with a vectorized version of the *same idea* — no new mathematics, just faster bookkeeping:

- **`find_generator`** (shared with Step 2) — the primitive root.
- **`power_residue_classes`** — walks the multiplicative group `g⁰, g¹, g², …` in vectorized NumPy blocks. That walk hits every nonzero residue exactly once, and the walk position mod r *is* the class. This is **one modular multiply per residue** — no per-element exponentiation, no square-and-multiply loop.
- **`run_stats`** — one vectorized `np.diff` pass for the longest run and the leading run.

Each function does one thing; the fast path is cross-checked against the naive `residue_colors` from Step 2 on p = 116,593 (run structure is invariant under relabeling colors, so the longest/leading runs must match *exactly*).

⏱️ *The record-prime call is ~30 seconds (≈10⁹ single modular multiplies, all in NumPy).*

Watch for: **max run = 18**, leading run = 1, boundary satisfied — which by the criterion validated above certifies **W(3,t) > (t−1)·969,397,381 + 1 for every t ≥ 19.**

In [ ]:
def power_residue_classes(p, r, block=1 << 22):
    """Array C with C[n-1] = class 0..r-1 of the integer n (n in 1..p-1)
    under the r-th-power-residue coloring, i.e. (discrete log of n) mod r.

    Same construction as the naive residue_colors, vectorized: the group
    walk g^0, g^1, g^2, ... visits every nonzero residue once, so we take
    it a block at a time -- g^(i..i+B) = g^i * (g^0..g^B) -- and scatter
    each residue's walk-position mod r into place. One modular multiply
    per residue; no per-element exponentiation."""
    g = find_generator(p)
    gpows = np.empty(block, dtype=np.int64)         # g^0, g^1, ..., g^(B-1)
    gpows[0] = 1
    for k in range(1, block):
        gpows[k] = gpows[k - 1] * g % p
    step = pow(g, block, p)                          # advance one block: * g^B
    classes = np.empty(p - 1, dtype=np.uint8)
    s = 1
    for i in range(0, p - 1, block):
        vals = gpows[: min(block, p - 1 - i)]
        if s != 1:
            vals = vals * s % p                      # g^i * g^k = g^(i+k)
        classes[vals - 1] = (np.arange(i, i + len(vals)) % r).astype(np.uint8)
        s = s * step % p
    return classes


def run_stats(classes):
    """(longest run, leading run) of equal consecutive entries, vectorized."""
    changes = np.flatnonzero(np.diff(classes))       # where the class changes
    edges = np.concatenate(([-1], changes, [classes.size - 1]))
    longest = int(np.diff(edges).max())
    leading = int(changes[0]) + 1 if changes.size else classes.size
    return longest, leading


# cross-check the fast path against the naive one (Step 2) on p = 116,593
ref = residue_colors(116593, 3)
assert run_stats(power_residue_classes(116593, 3)) == (max_run(ref),
                                                       leading_run(ref))
print("fast path agrees with the naive walk on p=116,593")

# the record prime
P = 969_397_381
classes = power_residue_classes(P, 3)
longest, leading = run_stats(classes)
same_ends = classes[0] == classes[-1]
print(f"record prime p={P}: max_run={longest}, leading_run={leading}")

boundary = 19 // 2 if same_ends else 18              # criterion bound for t=19
assert longest == 18 and leading == 1 and leading < boundary
print()
for t in range(19, 26):
    print(f"  W(3,{t}) > {(t - 1) * P + 1:,}")

## Step 5 — Does it say *no* when it should?

A verifier that only ever prints "valid" is worthless. Two demonstrations that this one rejects near-misses:

1. **Right at the boundary, fully brute-forced.** The prime p = 1033 gives a 3-coloring whose longest run is 6. Its certificate for t = 6 therefore *contains* a monochromatic 6-term progression — brute force locates it — so p = 1033 does **not** certify W(3,6). Bump to t = 7 and the very same construction is clean. The method flips from fail to pass exactly at the run length, and brute force agrees on both sides.
2. **The record prime is tight.** The coloring behind our records has a run of *exactly* 18 equal-class consecutive integers — an explicit monochromatic 18-term progression (common difference 1). So this prime does **not** certify W(3,18); it only starts working at t ≥ 19. We locate and print those 18 integers.

In [ ]:
# 1) boundary example, fully brute-force checked
p = 1033
cols = residue_colors(p, 3)
m = max_run(cols)
at_m = has_mono_ap_fast(build_certificate(p, 3, m, cols), m)
at_m1 = has_mono_ap_fast(build_certificate(p, 3, m + 1, cols), m + 1)
print(f"p={p}: longest run = {m}")
print(f"  t={m}: certificate CONTAINS a mono {m}-AP  -> W(3,{m}) NOT certified")
print(f"  t={m+1}: certificate is CLEAN              -> "
      f"W(3,{m+1}) > {m * p + 1:,} certified")
assert at_m and not at_m1   # has_mono_ap_fast returns True/False

# 2) the record prime is tight -- exhibit its run of 18
changes = np.flatnonzero(np.diff(classes))
edges = np.concatenate(([-1], changes, [classes.size - 1]))
runs = np.diff(edges)
i = int(runs.argmax())
start, length = int(edges[i] + 1), int(runs[i])   # classes[k] is integer k+1
lo_int = start + 1
assert (classes[start:start + length] == classes[start]).all()
print(f"\nrecord prime p={P}: longest run = {length} -- integers "
      f"{lo_int}..{lo_int + length - 1} all have class {int(classes[start])}")
print(f"  => an explicit monochromatic {length}-AP, so p={P} does NOT certify "
      f"W(3,{length}); it starts at t={length + 1}")
assert length == 18

## What a human still needs to check (the honest residue)

The computation above is now as verified as computation gets. What it does **not** establish by itself:

1. **That the previous bounds are what we say they are.** Check the r=3 column of Monroe's table: his paper ([JCMCC 128 (2026) 305–315](https://combinatorialpress.com/article/jcmcc/Volume%20128/new-lower-bounds-for-van-der-waerden-numbers-using-distributed-computing.pdf); [arXiv:1603.03301](https://arxiv.org/abs/1603.03301)) and the tally file `output.txt` in [github.com/hmonroe/vdw](https://github.com/hmonroe/vdw). His scan was exhaustive to ~969,396,749 and stopped in 2019 (the BOINC project died); our prime is the first valid one past that mark.
2. **That no recursion theorem shadows these cells.** For r ≥ 4, bounds implied by the Blankenship–Cummings–Taranchuk recursion ([EJC 69 (2018)](https://arxiv.org/abs/1705.09673)) dwarf any scan — we found (and discarded) three such shadowed "records" earlier the same day. For r = 3 at these lengths the recursion gives ~t·W(2,t) ≈ 10⁸–10¹¹·t… concretely, far below the scan values; and Xu-type recursions multiply *color counts*, giving nothing for prime r = 3. This is checkable arithmetic from the cited papers, and it is *why Monroe's own published tables include r = 3*.
3. **Community convention** — that extending a stopped exhaustive scan counts as the new best known. It does: it is precisely how the standing records were set and published, most recently December 2025.

**Recommended next step:** email Tim/Hugh Monroe (contact on the arXiv paper / GitHub repo) — the person who maintains exactly these tables, who published the standing values seven months ago, and who explicitly flagged the near-billion band as incompletely checked. The message is three sentences: *we continued your scan past 969,396,749; the attached notebook and prime list are the result; could you confirm?* Marijn Heule (CMU) is a natural second reader. This is a normal, welcome email in this community — certificate results are checked by recomputation, not refereeing.

## Post-mortem: the search strategy, honestly

This result came at the end of a one-day session that started with "pick a fun unsolved problem." The retrospective matters more than the records, because the *pattern* is reusable. The day, compressed:

1. **Ramsey numbers (classical cells: R(4,6), R(4,7), R(3,3,3,3)).** Simulated annealing over circulant then Cayley-graph colorings: reproduced every known extremal witness (Paley 17, the GF(16) 3-coloring, tight R(3,9)/R(4,5) colorings) and set nothing. Pivot: freezing a record graph and asking "does a one-vertex extension exist?" turns an unsearchable 2^630 space into an exactly-decidable 35-variable CSP. Result — a real one, though negative: **all 39 known record graphs for these cells, and every valid 1- and 2-edge modification of them (~700k variants), are provably non-extendable.** The records sit in deep isolated basins ≥ 3 edits from anything better; that is *why* they have stood for decades. An hour of stochastic search had flailed at what exact decision settled in milliseconds.
2. **The zipper trap.** We implemented the cyclic-zipper doubling method for van der Waerden bounds, validated it against every known zipped record (with full brute-force certificate checks) and against the reference C code — it caught a transcription bug via the paper's own worked example — and swept 66 never-checked candidate primes. Three passed: valid, novel certificates for W(8,23), W(6,24), W(8,21). **None are records**: a 2018 recursion theorem implies bounds 10⁵–10¹⁶× larger for r ≥ 4. Verify the *record-keeping*, not just the mathematics, before celebrating.
3. **The seam.** The same verification pass revealed where recursions do NOT reach: r = 2 and r = 3 at large t, where the standing records are literally "the largest prime an exhaustive scan reached before its distributed-computing project died in 2019." The r = 3 table cells sit at the scan cap — meaning valid primes are *dense* there — and nobody had checked a single prime past 969,396,749 in six years. The scanner found 14 valid primes in its first hour; roughly half of all primes checked were valid. The seven records fell out of the most trivial code written all day.

### What actually moved the needle

Every unit of progress came from moving the search **up a level**; none came from searching harder at the current level. The object-level search (annealing, modular exponentiation) was commodity. The outer loops — *which problem, which table, which cell, which method* — were driven by **reading**: papers, source code, tally files, and the sociology of which compute efforts had stopped and when. The decisive facts were things like a hard-coded `if (prime > 40000000) return 0;` in a 2016 C file, and a table caption saying "we only present bounds for r ≤ 4 since for more colors our bounds were beaten by the recursions." All of the alpha was in the field's metadata.

The reusable screening profile for "winnable by this kind of effort":
1. the claim is **certificate-checkable** (cheap verification is both the search's fitness function and its insurance policy);
2. records were set by **compute that visibly stopped** (dead BOINC project, 2013-era SAT solver, a 5-minute solver timeout in a 2017 paper) rather than by an idea;
3. **no theory shadow** — check the recursion/implication literature *first*;
4. an actively **maintained table** exists to land the result in.

Also for the record: the single biggest constant-factor win of the day (a ~100× early-abort restructuring of the candidate sweep) came from a human looking at a loop and asking whether it was really necessary to pay full price for candidates that fail in the first megabyte. The heuristic she cited was "never use a loop." The loop was innocent. The instinct was correct anyway.

## Appendix: provenance & artifacts

- All 14 valid primes and 78 record entries: `VDW_RECORDS.json` (this repo). The best-per-cell bounds all come from p = 969,397,381; the other 13 primes independently corroborate (each is its own certificate).
- Search/verification code as actually run: `code/` in this repo (`vdw.py`, `vdw_scan3.py`, `vdw_zip*.py` — the latter validated against R. Rabung & M. Lotts, *Improving the Use of Cyclic Zippers in Finding Lower Bounds for van der Waerden Numbers*, E-JC 19(2) #P35 (2012), and Monroe's reference implementation).
- Independent confirmation already performed: the record prime rescanned with a different primitive root (labels permute; run structure — max run 18, leading run 1 — reproduced exactly) plus 50,000-position scalar spot-checks of the vectorized arithmetic against Python's built-in `pow`.
- Session lab notebook with the full decision trail: `NOTES.md`.

*Prepared July 20, 2026. If you are a mathematician reading this because someone you once dated in grad school sent it to you: the authors apologize, and cell-by-cell execution takes about five minutes, most of it in Step 4.*